### SynthEval Benchmark guide

This notebook demonstrates how SynthEval can be used in dataset benchmarks for bulk evaluation and internal ranking of synthetic tabular datasets. This guides the user in selecting between several instances of synthetic data, for example produced during hyperparameter search, or from different generative models. This is a practical use case that reflects a decision that an analyst using synthetic data generation methods would want to make.

This example is based on the "Hepatitis C Virus (HCV) for Egyptian patients" available from UCI (https://archive.ics.uci.edu/dataset/503).

In [1]:
### Imports
import pandas as pd 
from syntheval import SynthEval

### Access datasets
df_train = pd.read_csv('example/hepatitis_train.csv')
df_test  = pd.read_csv('example/hepatitis_test.csv')

# The datasets can be supplied as a filepath or alternatively as a dictionary of dataframes i.e. {'data1': df_syn1, ...}
SYN_PATH = 'example/ex_data_dir/'   

In [2]:
### Dictionary of metric configuration, could practically be placed in a json file instead of taking up space in a script
metrics = {
    "corr_diff" : {"mixed_corr": True},
    "mi_diff"   : {},
    "ks_test"   : {"sig_lvl": 0.05, "n_perms": 1000},
    "p_mse"     : {"k_folds": 5, "max_iter": 1000, "solver": "liblinear"},
    "cls_acc"   : {"F1_type": "micro", "k_folds": 5},
    "dcr"       : {},
    "eps_risk"  : {},
    "mia_risk"  : {"num_eval_iter": 5},
    "att_discl" : {}
}

For the benchmark module the ranking strategy is a key parameter. The ranking strategy determines how the results from the different metrics are combined into a single ranking of the synthetic datasets. 
The options are:
- "summation": Uses default normalisation sums the normalised numbers. A higher number is better.
- "linear": Apply min-max scaling to the normalised columns, take the row sum as rank. A higher number is better.
- "quantile": The normalised numbers are converted to quantiles and then summed. A higher number is better.
- "normal": Map worst and best score to 0 and 1 respectively, everything else is 0.5. A higher number is better.

In [4]:
class_cat_col = ['Gender','Fever','Nausea/Vomting','Headache','Diarrhea','Fatigue & generalized bone ache','Jaundice','Epigastric pain','WBC','RBC','Plat','RNA Base','RNA 4','RNA 12','RNA EOT','RNA EF','Baselinehistological staging']
predict_class = 'Baselinehistological staging'

SE = SynthEval(df_train, holdout_dataframe=df_test, cat_cols=class_cat_col)

df_vals, df_rank = SE.benchmark(SYN_PATH, predict_class, rank_strategy='linear', **metrics)

Rich console is not supported in this environment. Defaulting to ascii console.


In [5]:
df_vals

corr_mat_diff       mutual_inf_diff        \
                                        value error           value error   
dataset                                                                     
a_hepatitis_sampling_baseline        2.466638   NaN        2.367049   NaN   
b_hepatitis_smote_baseline           0.410143   NaN        2.973446   NaN   
c_hepatitis_daib_baseline            2.071261   NaN        2.751713   NaN   
d_hepatitis_datasynthesizer_syn      0.404462   NaN        0.169534   NaN   
e_hepatitis_synthpop_syn_best        0.440553   NaN        0.263388   NaN   
f_hepatitis_CTGAN_syn_best           0.812515   NaN         1.87706   NaN   
g_hepatitis_ADSGAN_syn_best          0.830577   NaN        1.879755   NaN   
h_hepatitis_synthpop_1_syn           0.560022   NaN        0.198939   NaN   
i_hepatitis_BN_7_syn                  2.33441   NaN        2.219916   NaN   
j_hepatitis_CTGAN_9_syn              2.459151   NaN        2.801471   NaN   
k_hepatitis_ADSGAN_2_syn             3.361357   NaN        2.748846   NaN   

                                ks_tvd_stat           frac_ks_sigs        \
                                      value     error        value error   
dataset                                                                    
a_hepatitis_sampling_baseline      0.017106  0.002039          0.0   NaN   
b_hepatitis_smote_baseline         0.068173  0.010552     0.448276   NaN   
c_hepatitis_daib_baseline          0.262923  0.027716     0.931034   NaN   
d_hepatitis_datasynthesizer_syn    0.019911  0.002618     0.068966   NaN   
e_hepatitis_synthpop_syn_best      0.017943  0.002158          0.0   NaN   
f_hepatitis_CTGAN_syn_best         0.052371  0.008305      0.37931   NaN   
g_hepatitis_ADSGAN_syn_best        0.046837  0.006174     0.482759   NaN   
h_hepatitis_synthpop_1_syn         0.020103  0.002783          0.0   NaN   
i_hepatitis_BN_7_syn               0.137423  0.030898     0.448276   NaN   
j_hepatitis_CTGAN_9_syn            0.063528  0.011074     0.551724   NaN   
k_hepatitis_ADSGAN_2_syn           0.097899  0.013013     0.724138   NaN   

                                 avg_pMSE            ... median_DCR  \
                                    value     error  ...      error   
dataset                                              ...              
a_hepatitis_sampling_baseline    0.000401  0.000013  ...        NaN   
b_hepatitis_smote_baseline       0.001536  0.000026  ...        NaN   
c_hepatitis_daib_baseline        0.000001       0.0  ...        NaN   
d_hepatitis_datasynthesizer_syn  0.002759  0.000941  ...        NaN   
e_hepatitis_synthpop_syn_best    0.001021  0.000263  ...        NaN   
f_hepatitis_CTGAN_syn_best       0.000401  0.000013  ...        NaN   
g_hepatitis_ADSGAN_syn_best      0.000401  0.000013  ...        NaN   
h_hepatitis_synthpop_1_syn       0.001044  0.000443  ...        NaN   
i_hepatitis_BN_7_syn             0.057477  0.003004  ...        NaN   
j_hepatitis_CTGAN_9_syn          0.000401  0.000013  ...        NaN   
k_hepatitis_ADSGAN_2_syn         0.000401  0.000013  ...        NaN   

                                eps_identif_risk       priv_loss_eps        \
                                           value error         value error   
dataset                                                                      
a_hepatitis_sampling_baseline           0.536295   NaN      0.186349   NaN   
b_hepatitis_smote_baseline              0.848321   NaN      0.512459   NaN   
c_hepatitis_daib_baseline               0.705309   NaN      0.438787   NaN   
d_hepatitis_datasynthesizer_syn         0.689057   NaN      0.359697   NaN   
e_hepatitis_synthpop_syn_best           0.630553   NaN      0.289274   NaN   
f_hepatitis_CTGAN_syn_best              0.553629   NaN      0.217768   NaN   
g_hepatitis_ADSGAN_syn_best             0.578548   NaN      0.229686   NaN   
h_hepatitis_synthpop_1_syn              0.566631   NaN      0.219935   NaN   
i_hepatitis_BN_7_syn               

In [ ]:
# just to show how the scores have been min-max normalised.
df_rank

metric,corr_mat_diff,mutual_inf_diff,ks_tvd_stat,frac_ks_sigs,avg_pMSE,avg_F1_diff,avg_F1_diff_hout,median_DCR,eps_identif_risk,priv_loss_eps,att_discl_risk,rank,u_rank,p_rank,f_rank
dataset,,,,,,,,,,,,,,,
a_hepatitis_sampling_baseline,0.302587,0.216268,1.000000,1.000000,0.993037,0.841730,0.916877,0.566241,0.647191,0.767857,0.513241,7.765029,5.270499,2.494530,0.0
b_hepatitis_smote_baseline,0.998079,0.000000,0.792255,0.518519,0.973287,0.725516,0.909320,0.000000,0.000000,0.000000,0.000000,4.916974,4.916974,0.000000,0.0
c_hepatitis_daib_baseline,0.436301,0.079080,0.000000,0.000000,1.000000,0.000000,0.000000,0.749202,0.296629,0.173469,0.882181,3.616863,1.515381,2.101482,0.0
d_hepatitis_datasynthesizer_syn,1.000000,1.000000,0.988589,0.925926,0.952011,0.978756,0.984887,0.309951,0.330337,0.359694,0.124327,7.954477,6.830168,1.124309,0.0
e_hepatitis_synthpop_syn_best,0.987794,0.966528,0.996596,1.000000,0.982249,0.920334,1.000000,0.303315,0.451685,0.525510,0.118268,8.252280,6.853501,1.398779,0.0
f_hepatitis_CTGAN_syn_best,0.861999,0.391020,0.856542,0.592593,0.993037,1.000000,0.942065,0.352213,0.611236,0.693878,0.193223,7.487805,5.637257,1.850549,0.0
g_hepatitis_ADSGAN_syn_best,0.855891,0.390059,0.879054,0.481481,0.993037,0.927769,0.861461,0.303341,0.559551,0.665816,0.201975,7.119436,5.388752,1.730683,0.0
h_hepatitis_synthpop_1_syn,0.947391,0.989513,0.987808,1.000000,0.981861,0.915023,0.886650,0.308384,0.584270,0.688776,0.181329,8.471003,6.708246,1.762757,0.0
i_hepatitis_BN_7_syn,0.347306,0.268742,0.510543,0.518519,0.000000,0.775873,0.931990,1.000000,0.970787,0.954082,1.000000,7.277841,3.352973,3.924868,0.0
